# Evaluation

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

In [ ]:
reference_path = Path("/gdata2/simon/gprof_ir/conus/mrms/")
results = {
    "P-UNet": Path("/gdata2/simon/gprof_ir/conus/punet//"),
    "PERSIANN CCS": Path("/gdata2/simon/gprof_ir/conus/ccs/"),
    "PERSIANN PDIR-Now": Path("/gdata2/simon/gprof_ir/conus/pdir_now/"),
    "GPROF-IR (GMI, 3)": Path("/gdata2/simon/gprof_ir/conus/gprof_ir_gmi_3//"),
    "IMERG": Path("/gdata2/simon/gprof_ir/conus/imerg/"),
}

In [ ]:
from datetime import datetime
from typing import Dict

from satrain import metrics
from gprof_ir import metrics as metrics_ir
from tqdm import tqdm

metric_classes = [
    metrics.Bias,
    metrics.MAE,
    metrics.MSE,
    metrics.SMAPE,
    metrics.CorrelationCoef,
]
spatial_metric_classes = [
    metrics_ir.BiasSpatial,
    metrics_ir.MAESpatial
]
    

def get_date(path):
    parts = path.name.split("_")
    return datetime.strptime(parts[-1][:-3], "%Y%m%d%H%M%S")
    
def evaluate(
    reference_path: Path,
    results: Dict[str, Path]
):
    ref_files = {
        get_date(path): path for path in reference_path.glob("*.nc")
    }
    res_files = {}
    for name, path in results.items():
        res_files[name] = {
            get_date(path): path for path in path.glob("*.nc")
        }
    matched_times = set.intersection(set(ref_files), *[set(files) for files in res_files.values()])

    metrics = {
        name: [metric_class() for metric_class in metric_classes] for name in res_files
    }
    spatial_metrics = {
        name: [metric_class() for metric_class in spatial_metric_classes] for name in res_files
    }
    matched_times = list(matched_times)

    for time in tqdm(matched_times[:10]):
        ref_data = xr.load_dataset(ref_files[time])
        sp_ref = ref_data.surface_precip.data
        rqi = ref_data.radar_quality_index.data
        sp_ref[~(0.5 < rqi)] = np.nan

        lons = ref_data.longitude.data
        lats = ref_data.latitude.data
        lons, lats = np.meshgrid(lons, lats, indexing="xy")

        try:
            surface_precip = {}
            for name, files in res_files.items():
                with xr.open_dataset(files[time]) as data:
                    surface_precip[name] = data.surface_precip.compute().data.squeeze()
        except Exception:
            print(name, data.surface_precip.shape)
            continue
    
        valid = (0.0 <= sp_ref)
        for name, sp_ret in surface_precip.items():
            valid *= (0.0 <= sp_ret)

        for name, sp_ret in surface_precip.items():
            for metric in metrics[name]:
                metric.update(sp_ret[valid], sp_ref[valid])
            for metric in spatial_metrics[name]:
                metric.update(lons[valid], lats[valid], sp_ret[valid], sp_ref[valid])

    results = {}
    for name, mtrcs in metrics.items():
        res = []
        for metric in mtrcs:
            res.append(metric.compute())
        for metric in spatial_metrics[name]:
            res.append(metric.compute())
        results[name] = xr.merge(res)

    return results


In [ ]:
res = evaluate(reference_path=reference_path, results=results)

In [ ]:
plt.pcolormesh(res["P-UNet"].bias_spatial, vmin=-10, vmax=50)
plt.xlim(40, 120)
plt.ylim(100, 160)
plt.colorbar()

In [ ]:
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LogNorm

def plot_results(
    reference_path: Path,
    results: Dict[str, Path],
    ind
):
    ref_files = {
        get_date(path): path for path in reference_path.glob("*.nc")
    }
    res_files = {}
    for name, path in results.items():
        res_files[name] = {
            get_date(path): path for path in path.glob("*.nc")
        }
    matched_times = set.intersection(set(ref_files), *[set(files) for files in res_files.values()])

    metrics = {
        name: [metric_class() for metric_class in metric_classes] for name in res_files
    }
    matched_times = list(matched_times)
    time = matched_times[ind]

    ref_data = xr.load_dataset(ref_files[time])
    sp_ref = ref_data.surface_precip.data
    rqi = ref_data.radar_quality_index.data
    sp_ref[~(0.5 < rqi)] = np.nan

    surface_precip = {}
    for name, files in res_files.items():
        print(name, files[time])
        with xr.open_dataset(files[time]) as data:
            surface_precip[name] = data.surface_precip.compute().data.squeeze()

    valid = (0.0 <= sp_ref)
    for name, sp_ret in surface_precip.items():
        valid *= (0.0 <= sp_ret)

    for name, sp_ret in surface_precip.items():
        for metric in metrics[name]:
            metric.update(sp_ret[valid], sp_ref[valid])

    results = {}
    for name, mtrcs in metrics.items():
        res = []
        for metric in mtrcs:
            res.append(metric.compute())
        results[name] = xr.merge(res)

    n_panels = len(results) + 1
    
    gs = GridSpec(n_panels, 2, width_ratios=[1.0, 0.075])
    fig = plt.figure(figsize=(7, 3 * n_panels))

    norm = LogNorm(1e-1, 5e1)
    ax = fig.add_subplot(gs[0, 0])
    ax.imshow(sp_ref)
    ax.set_title("Reference")

    for ax_ind, (name, sp_ret) in enumerate(surface_precip.items()):
        ax = fig.add_subplot(gs[ax_ind + 1, 0])
        ax.imshow(sp_ret, norm=norm)
        ax.set_title(name)

    return fig

In [ ]:
plot_results(reference_path=reference_path, results=results, ind=30)